# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library. The dataset is described using the Croissant schema and includes rich metadata, multiple record sets, and standardized field references via `@id`.

## Dataset Source
The dataset schema is accessible at:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

This notebook is organized in the following steps:
- Data Loading
- Data Overview
- Data Extraction
- Exploratory Data Analysis (EDA)
- Visualization
- Conclusion

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Dataset metadata access as attributes (not by subscripting)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview

Explore the available record sets and their structure. We'll retrieve the list of record sets, and for each, print the `@id`, name, and the field IDs associated with each record set. This helps identify how to target the data of interest and refer to all components strictly by their `@id` as per the Croissant schema.

In [ ]:
# List all record sets with their @id and all fields and columns they contain.
for record_set in dataset.metadata.record_sets:
    print(f"Record Set @id: {record_set['@id']}")
    print(f"  Name: {record_set.get('name','(no name)')}")
    field_ids = [field['@id'] for field in record_set.get('field', [])]
    print(f"  Field @ids: {field_ids}")
    # Print columns per field if present
    for field in record_set.get('field', []):
        columns = field.get('column', [])
        col_ids = [c['@id'] for c in columns] if columns and isinstance(columns, list) else [columns['@id']] if isinstance(columns, dict) else []
        print(f"    Field {field['@id']} columns: {col_ids}")
    print('-'*60)

## 3. Data Extraction

Extract data for selected record sets into Pandas DataFrames for analysis. The `@id` for each record set is used to query and load records from the dataset. The example below loads all available record sets; you can limit or select specific ones as needed.

In [ ]:
# Identify all record set @ids
record_sets = [rs['@id'] for rs in dataset.metadata.record_sets]
print("Found record set @ids:", record_sets)

dataframes = {}
for record_set_id in record_sets:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Columns: {df.columns.tolist()}")
    print(df.head(2))
    print("-")

# For demonstration, pick the first record set for further processing
main_record_set_id = record_sets[0] if record_sets else None
if main_record_set_id is not None:
    print(f"Selected record set for analysis: {main_record_set_id}")
    print("Columns in this record set:", dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Apply typical EDA steps: filter records, normalize a numeric field, analyze groupings by a categorical variable. All field and column references use their Croissant `@id` values.

Below, replace the placeholder field IDs with those printed in the overview above for your specific analysis (example field IDs are shown as placeholders).

In [ ]:
# You may need to adjust field IDs based on the printed overview above.
import numpy as np

df = dataframes[main_record_set_id]
# Suppose 'gad7_total_score' is a field @id present as a column, adjust as needed.
numeric_field = 'gad7_total_score'  # <-- Replace with actual @id from overview step
group_field = 'gender'  # <-- Replace with actual @id

if numeric_field in df.columns:
    # Remove outliers (example: scores above 21 for GAD-7 (the max possible))
    filtered_df = df[(df[numeric_field].apply(pd.to_numeric, errors='coerce') <= 21)]
    print(f"Number of records after filtering {numeric_field} to <= 21: {len(filtered_df)}")

    # Normalization
    filtered_df[numeric_field+'_normalized'] = (
        filtered_df[numeric_field].astype(float) - filtered_df[numeric_field].astype(float).mean()
    ) / filtered_df[numeric_field].astype(float).std()

    print(filtered_df[[numeric_field, numeric_field+'_normalized']].head())

    # Group by category (e.g., gender)
    if group_field in filtered_df.columns:
        grouped = filtered_df.groupby(group_field)[numeric_field].agg(['count','mean','std'])
        print("Statistics by group:")
        print(grouped)
else:
    print(f"Field {numeric_field} not found in columns of record set {main_record_set_id}.")

## 5. Visualization

Visualize the distribution of a numeric mental health score (e.g., GAD-7 total score). This step demonstrates basic exploratory visualization using Pandas and Matplotlib.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna().astype(float), bins=range(0, 25), kde=True)
    plt.title(f"Histogram of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if group_field in df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=df[group_field], y=df[numeric_field].astype(float))
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()
else:
    print(f"Field {numeric_field} is not available in the current DataFrame for visualization.")

## 6. Conclusion

This notebook demonstrated how to use the `mlcroissant` library to load and process a FAIR² AI-ready dataset, referencing all entities by their Croissant `@id` as best practice. We've loaded metadata, explored record set/field structures, visualized data distributions, and performed simple grouped analyses.

- Use Croissant `@id`s in all code for robust, schema-driven data access.
- This dataset makes it easy to scale analysis and adapt processing logic as the underlying schema evolves.

Refer to the [FAIR² schema documentation](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json) or use `dataset.metadata` methods for further inspection.